# DFKD/OOD strict ternary KD end-to-end

Leakage-safe transfer uses only a frozen teacher, Gaussian noise, unlabeled SVHN, and DeepInversion synthetic images. CIFAR-10 train and validation are prohibited. The last cell is the sole locked-test operation.

In [1]:
from pathlib import Path
import json, subprocess, sys
ROOT=Path('/home/vu-lab03-pc17/ATDL-1'); PIPELINE=ROOT/'scripts/run_dfkd_ood_pipeline.py'; OUT=ROOT/'dfkd_ood_2026'
assert PIPELINE.is_file()
print('SVHN labels are ignored. No target train/validation loader is allowed.')

SVHN labels are ignored. No target train/validation loader is allowed.


## Fixed research protocol

Manual, literature-backed progression: teacher-pseudo-class-balanced Gaussian baseline; teacher-pseudo-class-balanced SVHN arbitrary transfer; then DeepInversion BN-statistics-matched synthetic replay plus SVHN. The final direct-to-ternary candidate is selected solely by held-out transfer KD loss. No AutoML, sweep, or test-based decision is used.

In [2]:
subprocess.run([sys.executable,str(PIPELINE),'--epochs','35'],cwd=ROOT,check=True)
protocol=json.loads((OUT/'results/protocol.json').read_text()); print(json.dumps(protocol,indent=2))

DFKD pipeline complete; test= False


{
  "objective": "DFKD/OOD strict ternary QAT without CIFAR-10 training or validation data",
  "literature": [
    "Nayak et al. 2020 arbitrary transfer sets, arXiv:2011.09113",
    "Yin et al. 2020 DeepInversion, CVPR",
    "Choi et al. 2020 data-free network quantization, CVPRW",
    "Liu et al. 2024 small-scale DFKD, CVPR"
  ],
  "fixed_protocol": {
    "baseline": "balanced Gaussian, 5 epochs",
    "final": "balanced SVHN + DeepInversion BN-stat synthetic, 35 epochs",
    "selection": "transfer-held-out KD loss only"
  },
  "transfer_sources": [
    "Gaussian noise",
    "SVHN train split (unlabeled)",
    "DeepInversion synthetic teacher-guided samples"
  ],
  "provenance": {
    "svhn_url": "http://ufldl.stanford.edu/housenumbers/",
    "labels": "ignored"
  },
  "final_checkpoint": "dfkd_ood_2026/checkpoints/dfkd_di_svhn_ternary_best.pth"
}


## Locked final evaluation

Run only after the frozen pipeline above succeeds. It evaluates once, saves immutable per-class and confusion-matrix diagnostics, and refuses overwrites.

In [3]:
locked_path=OUT/'results/locked_final_test.json'
if not locked_path.exists():
 subprocess.run([sys.executable,str(PIPELINE),'--epochs','35','--locked-final-test'],cwd=ROOT,check=True)
print(json.dumps(json.loads(locked_path.read_text()),indent=2))

{
  "checkpoint": "dfkd_ood_2026/checkpoints/dfkd_di_svhn_ternary_best.pth",
  "test_accuracy": 0.1459999978542328,
  "per_class_accuracy": {
    "airplane": 0.13699999451637268,
    "automobile": 0.039000000804662704,
    "bird": 0.2669999897480011,
    "cat": 0.15800000727176666,
    "deer": 0.0,
    "dog": 0.46000000834465027,
    "frog": 0.3019999861717224,
    "horse": 0.008999999612569809,
    "ship": 0.003000000026077032,
    "truck": 0.08500000089406967
  },
  "confusion_matrix": [
    [
      137,
      38,
      83,
      198,
      1,
      479,
      20,
      2,
      0,
      42
    ],
    [
      27,
      39,
      169,
      362,
      2,
      342,
      16,
      2,
      0,
      41
    ],
    [
      28,
      7,
      267,
      115,
      1,
      361,
      206,
      0,
      0,
      15
    ],
    [
      2,
      6,
      260,
      158,
      2,
      496,
      66,
      1,
      1,
      8
    ],
    [
      9,
      3,
      224,
      79,
      0,
      